<a href="https://colab.research.google.com/github/ChandrikaImmadi/GenAI-Tasks/blob/main/Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai faiss-cpu tiktoken pypdf chromadb

In [ ]:
import os
import openai
import faiss
import numpy as np
import tiktoken
from pypdf import PdfReader
from typing import List
from google.colab import files


In [ ]:
openai.api_key = "API_KEY"

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    tokens = text.split()
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = " ".join(tokens[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
def extract_text_from_file(file_path):
    if file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    elif file_path.endswith(".txt") or file_path.endswith(".md"):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type")

In [ ]:
uploaded = files.upload()
documents = []
for fname in uploaded.keys():
    text = extract_text_from_file(fname)
    documents.append(text)

In [ ]:
dimension = 1536  # OpenAI embedding dimension
index = faiss.IndexFlatL2(dimension)
doc_chunks = []
embeddings = []

# Initialize the OpenAI client for the new API version
client = openai.OpenAI(api_key=openai.api_key)

for doc in documents:
    chunks = chunk_text(doc)
    for chunk in chunks:
        response = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )
        vector = np.array(response.data[0].embedding).astype('float32')
        index.add(np.array([vector]))
        doc_chunks.append(chunk)
        embeddings.append(vector)

print(f"Stored {len(doc_chunks)} chunks in FAISS index.")

In [ ]:
def answer_query(query, top_k=5):
    # Embed query
    response = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    )
    query_vector = np.array(response.data[0].embedding).astype('float32')

    # Search FAISS
    distances, indices = index.search(np.array([query_vector]), top_k)
    retrieved_chunks = [doc_chunks[i] for i in indices[0]]

    # Build prompt
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""

    # Get answer from ChatCompletion
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a knowledgeable assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
print(answer_query("Summarize the main points of the documents."))